In [1]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import ast
from sklearn.metrics import f1_score, classification_report
from torch.amp import autocast, GradScaler
import random
import os
import numpy as np


In [2]:

def set_seed(seed=42):
    """Locks all random number generators for exact reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)


set_seed(42)


### Dataset generation

In [3]:
# Load all the data 

# Column names for binary classification PCL data
column_names = ['par_id', 'art_id', 'keyword', 'country_code', 'text', 'label']

# Reading the data, skipping the first 4 lines of disclaimer
# delimiter is set to \t for tab-separated values
df_pcl = pd.read_csv('data/dontpatronizeme_pcl.tsv', 
                 sep='\t', 
                 skiprows=4, 
                 names=column_names, 
                 index_col=False,
                 quoting=3)


# data load for multi label classification

span_columns = [
    'par_id', 'art_id', 'text', 'keyword', 'country_code', 
    'span_start', 'span_finish', 'span_text', 'pcl_category', 'num_annotators'
]

# Load the data
df_categories= pd.read_csv('data/dontpatronizeme_categories.tsv', 
                       sep='\t', 
                       skiprows=4, 
                       names=span_columns, 
                       index_col=False,
                       quoting=3) # quoting=3 tells pandas to ignore quotes to avoid splitting text mid-sentence

# Training labels
df_train_labels = pd.read_csv('data/train_semeval_parids-labels.csv', index_col=False)

df_dev_labels = pd.read_csv('data/dev_semeval_parids-labels.csv', index_col=False)
                             


In [4]:
# Binary PCL classifcation
# Create the new binary column 'pcl_presence'
df_pcl['pcl_presence'] = df_pcl['label'].apply(lambda x: 0 if x in [0, 1] else 1)

import ast
df_train_labels['label'] = df_train_labels['label'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

df_dev_labels['label'] = df_dev_labels['label'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

In [5]:
import html
import re

# Clean data 

def clean_pcl_text(text):
    text = str(text)
    
    # Convert HTML entities back to normal characters (e.g., &amp; -> &)
    text = html.unescape(text)
    
    # Strip out remaining HTML tags (e.g., <br>, <i>)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 3Fix multiple spaces that might have been created by removing tags
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply the cleaning function to the dataset

df_pcl['text'] = df_pcl['text'].apply(clean_pcl_text)

# Verify the cleaning worked
html_pattern = r'<[^>]+>'
entity_pattern = r'&[a-z]+;'

remaining_html = df_pcl['text'].str.contains(html_pattern, regex=True).sum()
remaining_entities = df_pcl['text'].str.contains(entity_pattern, regex=True).sum()

print(f"Remaining HTML tags and entities:{remaining_html} , {remaining_entities}")


Remaining HTML tags and entities:0 , 0


In [6]:
# merge train and dev labels with pcl presence info

import pandas as pd
import ast


def prepare_final_dataset(df_labels, df_main_text):
    
    # Merge to get the text, keyword, and country code based on par_id
    df_merged = pd.merge(df_labels, df_main_text[['par_id', 'text', 'keyword', 'country_code',"pcl_presence"]], 
                         on='par_id', how='left')
    
    # Format the input text (Keyword + Country + Text)
    # Using RoBERTa/DeBERTa's separator token </s> 
    df_merged['model_input'] = (
        df_merged['keyword'].astype(str) + " </s> " + 
        df_merged['country_code'].astype(str) + " </s> " + 
        df_merged['text'].astype(str)
    )
        
    return df_merged[['par_id', 'model_input', 'pcl_presence']]


train_val_data = prepare_final_dataset(df_train_labels, df_pcl)
dev_data = prepare_final_dataset(df_dev_labels, df_pcl)


# Split train data into train and validate
train_data, val_data = train_test_split(
    train_val_data, 
    test_size=0.15, # 15% goes to validation
    stratify=train_val_data['pcl_presence'], 
    random_state=42 # 
)

train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print(train_data.head(3))

   par_id                                        model_input  pcl_presence
0    1748  hopeless </s> lk </s> Contrary to such measure...             0
1    3205  immigrant </s> au </s> "He said the city has b...             0
2    4237  disabled </s> au </s> "On the same page as the...             0


### Model Specs

In [7]:
model_name = 'roberta-base'


class PCLMultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256):
        self.texts = df['model_input'].tolist()
        self.binary_labels = df['pcl_presence'].tolist()
        # self.multi_labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_length,
            padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        
        binary_label = torch.tensor([self.binary_labels[idx]], dtype=torch.float32)
        
        # m_label = self.multi_labels[idx]
        # if isinstance(m_label, str):
        #     m_label = ast.literal_eval(m_label)
        # multi_label_tensor = torch.tensor(m_label, dtype=torch.float32)

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'binary_labels': binary_label,
            # 'multi_labels': multi_label_tensor
        }

# class MultiTaskRobertaPCL(nn.Module):
#     def __init__(self, model_name='roberta-base', num_categories=7):
#         super(MultiTaskRobertaPCL, self).__init__()
        
#         self.roberta = AutoModel.from_pretrained(model_name)
#         hidden_size = self.roberta.config.hidden_size 
#         self.dropout = nn.Dropout(0.3)

#         # 2 heads - one for binary classfication , another for categorical
        
#         self.binary_head = nn.Linear(hidden_size, 1)
#         self.category_head = nn.Linear(hidden_size, num_categories)

#     def forward(self, input_ids, attention_mask):
#         outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        
#         # Extract the [CLS] token representation
#         pooled_output = outputs.last_hidden_state[:, 0, :]
#         pooled_output = self.dropout(pooled_output)
        
#         binary_logits = self.binary_head(pooled_output)
#         category_logits = self.category_head(pooled_output)
        
#         return binary_logits, category_logits



class SingleTaskRobertaPCL(nn.Module):
    def __init__(self, model_name=model_name): 
        super().__init__()
        self.roberta = AutoModel.from_pretrained(model_name)
        hidden_size = self.roberta.config.hidden_size
        self.dropout = nn.Dropout(0.3)
        
        # ONLY ONE HEAD
        self.binary_head = nn.Linear(hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :] 
        pooled_output = self.dropout(pooled_output)
        
        bin_logits = self.binary_head(pooled_output)
        return bin_logits # No cat_logits!


In [8]:
# Initialize Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Build Datasets
train_dataset = PCLMultiTaskDataset(train_data, tokenizer)
val_dataset = PCLMultiTaskDataset(val_data, tokenizer)
dev_dataset = PCLMultiTaskDataset(dev_data, tokenizer)

# Calculate weights to fix the 90/10 Imbalance
class_counts = train_data['pcl_presence'].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = [class_weights[label] for label in train_data['pcl_presence']]

#  Initialize Sampler
sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=len(sample_weights), 
    replacement=True
)

# Build DataLoaders
train_loader = DataLoader(train_dataset, batch_size=8, sampler = sampler,drop_last=True)
# train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=8,shuffle=False )
dev_loader = DataLoader(dev_dataset, batch_size=8, shuffle=False)


In [9]:
#remove leftover cache from previous runs
torch.cuda.empty_cache()

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Initialize Model 
model = SingleTaskRobertaPCL().to(device)

# Standard BCE Loss
binary_criterion = nn.BCEWithLogitsLoss()
multi_criterion = nn.BCEWithLogitsLoss()

# Optimiser
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
scaler = GradScaler()


Device: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Training and validation

In [10]:

os.makedirs("models", exist_ok=True)

EPOCHS = 3
w_bin = 1.0
w_multi = 0.0 
# model_save_path = "models/best_roberta_model.pt" # path to save model

# Variables to model across all epochs
best_overall_val_f1 = 0.0
locked_optimal_thresh = 0.5

for epoch in range(EPOCHS):
    # Training
    model.train()
    total_loss = 0
    print(f"\n--- Starting Epoch {epoch + 1}/{EPOCHS} ---")
    
    for step, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        bin_labels = batch['binary_labels'].to(device)
        # multi_labels = batch['multi_labels'].to(device)
        
        optimizer.zero_grad()
        
        with autocast(device_type="cuda", dtype=torch.float16):
            bin_logits = model(input_ids, attention_mask)
            joint_loss = binary_criterion(bin_logits, bin_labels)
 
            
        scaler.scale(joint_loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += joint_loss.item()
        
        if (step + 1) % 200 == 0:
            print(f"   Batch {step + 1}/{len(train_loader)} - Loss: {joint_loss.item():.4f}")
            
    print(f"Epoch {epoch + 1} Train Loss: {total_loss / len(train_loader):.4f}")
    
    # validation on 15% of the data
    
    print("validation to get best threshold")
    model.eval() 
    
    val_probs, val_labels = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            bin_logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            probs = torch.sigmoid(bin_logits)
            val_probs.extend(probs.cpu().numpy())
            val_labels.extend(batch['binary_labels'].numpy())
            
    y_val_true = np.array(val_labels).flatten()
    y_val_probs = np.array(val_probs).flatten()
    
    epoch_best_f1, epoch_best_thresh = 0.0, 0.5
    for thresh in np.arange(0.1, 0.9, 0.01):
        preds = (y_val_probs >= thresh).astype(int)
        current_f1 = f1_score(y_val_true, preds, pos_label=1, zero_division=0)
        if current_f1 > epoch_best_f1:
            epoch_best_f1, epoch_best_thresh = current_f1, thresh
            
    print(f"Epoch {epoch + 1} Peak Val F1: {epoch_best_f1:.4f} (at threshold {epoch_best_thresh:.2f})")
    
    # --- CHECKPOINT SAVING LOGIC ---
    if epoch_best_f1 > best_overall_val_f1:
        print(f"F1 Improved from {best_overall_val_f1:.4f} to {epoch_best_f1:.4f}")
        best_overall_val_f1 = epoch_best_f1
        locked_optimal_thresh = epoch_best_thresh
        # torch.save(model.state_dict(), model_save_path)
    else:
        print(f"F1 did not improve")




--- Starting Epoch 1/3 ---
   Batch 200/889 - Loss: 0.5450
   Batch 400/889 - Loss: 0.4821
   Batch 600/889 - Loss: 0.0740
   Batch 800/889 - Loss: 0.1590
Epoch 1 Train Loss: 0.3865
validation to get best threshold
Epoch 1 Peak Val F1: 0.4956 (at threshold 0.89)
F1 Improved from 0.0000 to 0.4956

--- Starting Epoch 2/3 ---
   Batch 200/889 - Loss: 0.5949
   Batch 400/889 - Loss: 0.6892
   Batch 600/889 - Loss: 0.0462
   Batch 800/889 - Loss: 0.0012
Epoch 2 Train Loss: 0.2190
validation to get best threshold
Epoch 2 Peak Val F1: 0.5369 (at threshold 0.26)
F1 Improved from 0.4956 to 0.5369

--- Starting Epoch 3/3 ---
   Batch 200/889 - Loss: 0.0039
   Batch 400/889 - Loss: 0.0007
   Batch 600/889 - Loss: 0.0007
   Batch 800/889 - Loss: 0.0010
Epoch 3 Train Loss: 0.1397
validation to get best threshold
Epoch 3 Peak Val F1: 0.5736 (at threshold 0.76)
F1 Improved from 0.5369 to 0.5736


### Evaluation 

In [14]:
# Test on official dev set

print("Official Dev set")
# model.load_state_dict(torch.load(model_save_path))
model.eval()

dev_probs, dev_labels = [], []

with torch.no_grad():
    for batch in dev_loader:
        bin_logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        probs = torch.sigmoid(bin_logits)
        dev_probs.extend(probs.cpu().numpy())
        dev_labels.extend(batch['binary_labels'].numpy())
        
y_dev_true = np.array(dev_labels).flatten()
y_dev_probs = np.array(dev_probs).flatten()

# Apply the threshold from validation set
final_dev_preds = (y_dev_probs >= 0.5).astype(int)
final_dev_f1 = f1_score(y_dev_true, final_dev_preds, pos_label=1, zero_division=0)

print(f"\nFinal F1 : {final_dev_f1:.4f}")
print("\n--- FINAL CLASSIFICATION REPORT ---")
print(classification_report(y_dev_true, final_dev_preds, target_names=['Non-PCL (0)', 'PCL (1)']))

Official Dev set

Final F1 : 0.5965

--- FINAL CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

 Non-PCL (0)       0.97      0.94      0.95      1895
     PCL (1)       0.53      0.68      0.60       199

    accuracy                           0.91      2094
   macro avg       0.75      0.81      0.77      2094
weighted avg       0.92      0.91      0.92      2094



In [12]:
#### Results 

### Removing one head for Roberta Large 

##=================================
        # --- Starting Epoch 1/3 ---
        #    Batch 200/889 - Loss: 0.8499
        #    Batch 400/889 - Loss: 1.0622
        #    Batch 600/889 - Loss: 0.3298
        #    Batch 800/889 - Loss: 0.0028
        # Epoch 1 Train Loss: 0.4441
        # validation to get best threshold
        # Epoch 1 Peak Val F1: 0.5409 (at threshold 0.88)
        # F1 Improved from 0.0000 to 0.5409
        
        # --- Starting Epoch 2/3 ---
        #    Batch 200/889 - Loss: 0.0039
        #    Batch 400/889 - Loss: 0.0015
        #    Batch 600/889 - Loss: 0.6855
        #    Batch 800/889 - Loss: 0.0023
        # Epoch 2 Train Loss: 0.2284
        # validation to get best threshold
        # Epoch 2 Peak Val F1: 0.5664 (at threshold 0.55)
        # F1 Improved from 0.5409 to 0.5664
        
        # --- Starting Epoch 3/3 ---
        #    Batch 200/889 - Loss: 0.0004
        #    Batch 400/889 - Loss: 0.0031
        #    Batch 600/889 - Loss: 0.7060
        #    Batch 800/889 - Loss: 0.0002
        # Epoch 3 Train Loss: 0.1062
        # validation to get best threshold
        # Epoch 3 Peak Val F1: 0.5500 (at threshold 0.38)
        # F1 did not improve

# Official Dev set

# Final F1 : 0.5940

# --- FINAL CLASSIFICATION REPORT ---
#               precision    recall  f1-score   support

#  Non-PCL (0)       0.96      0.95      0.95      1895
#      PCL (1)       0.55      0.64      0.59       199

#     accuracy                           0.92      2094
#    macro avg       0.76      0.79      0.77      2094
# weighted avg       0.92      0.92      0.92      2094


## Removing thresholds and checking on threshold


